# Monitoring & Detection Audit

This notebook queries the OCP audit SQLite datastore to present findings for:

- **OCP-24 — OpenShift Usage Monitoring**: Use of OpenShift, including unapproved usage, is monitored. Evidence on-cluster: cluster-monitoring operator install posture, Prometheus / Alertmanager / user-workload-monitoring enablement, alerting rules, and any external monitoring agents (e.g. Datadog).
- **OCP-25 — Configuration Drift Detection**: Drift detection tooling and policies are implemented. Evidence on-cluster: MachineConfigPool state, ClusterOperator availability, GitOps (Argo CD / Flux) sync status, Compliance Operator scan results, and node / operator version consistency across the cluster.
- **OCP-26 — Centralized Logging, Auditing & Retention**: A unified and secure system for collecting, aggregating, and retaining all security and operational logs from across the entire environment is established. Evidence on-cluster: API server audit profile + retention, ClusterLogging operator + log forwarder pipelines, audit log forwarding, log retention, and LokiStack configuration.

In [ ]:
import os
import re
import sys

import pandas as pd

# Ensure the repo root is importable before loading project modules.
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from notebook_style import bootstrap, style_table  # noqa: E402

print("python:", sys.executable)
print("cwd:", os.getcwd())

# bootstrap() adds ../datastore to sys.path, which is required before
# importing schema.models below.
session, engine = bootstrap()

from schema.models import (  # noqa: E402
    Cluster,
    ConfigurationDriftStatus,
    MonitoringAuditLogging,
)

print("Connected to:", engine.url)

## Cluster Inventory

In [ ]:
df_clusters = pd.read_sql(
    session.query(
        Cluster.id,
        Cluster.cluster_name,
        Cluster.cluster_context,
        Cluster.cluster_server,
    ).statement,
    engine,
)
print(f"{len(df_clusters)} cluster(s) in dataset")
style_table(df_clusters)

---
## OCP-24: OpenShift Usage Monitoring

*Use of OpenShift, including unapproved usage, is monitored.*

On-cluster evidence is the install posture and configuration of the **cluster-monitoring** stack (Prometheus, Alertmanager, user-workload-monitoring), any **external monitoring** agent (e.g. Datadog), and the **alerting** rule / receiver configuration. Record types in scope: `operator` (cluster-monitoring), `monitoring_config`, `user_workload_monitoring`, `external_monitoring`, `alerting_rules`, `alertmanager`.

### Monitoring records (per cluster)

In [ ]:
OCP24_RECORD_TYPES = (
    "operator",
    "monitoring_config",
    "user_workload_monitoring",
    "external_monitoring",
    "alerting_rules",
    "alertmanager",
)

df_monitoring = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        MonitoringAuditLogging.record_type,
        MonitoringAuditLogging.component_name,
        MonitoringAuditLogging.status,
        MonitoringAuditLogging.namespace,
        MonitoringAuditLogging.detail_1,
        MonitoringAuditLogging.detail_2,
        MonitoringAuditLogging.detail_3,
        MonitoringAuditLogging.detail_4,
    )
    .join(Cluster, MonitoringAuditLogging.cluster_id == Cluster.id)
    .filter(MonitoringAuditLogging.record_type.in_(OCP24_RECORD_TYPES))
    .filter(
        ~(
            (MonitoringAuditLogging.record_type == "operator")
            & (MonitoringAuditLogging.component_name == "cluster-logging")
        )
    )
    .order_by(Cluster.cluster_name, MonitoringAuditLogging.record_type)
    .statement,
    engine,
)
style_table(df_monitoring, caption="OCP-24: Raw monitoring records")

### OCP-24: Compliance flags

Per-cluster flags derived from the raw records. A cluster is considered **compliant** when:

- `cluster_monitoring_installed` — `operator` row for `cluster-monitoring` has `status = installed`.
- `prometheus_enabled` — `detail_1` on the `cluster-monitoring` operator row contains `prometheus=enabled`.
- `alertmanager_enabled` — `detail_2` on the `cluster-monitoring` operator row contains `alertmanager=enabled`.
- `user_workload_monitoring_enabled` — `detail_3` on the `cluster-monitoring` operator row contains `user-workload-monitoring=true`.

Only clusters that are missing one or more flags are displayed.

In [ ]:
def _detail_has(value, needle):
    if value is None:
        return False
    return needle in str(value)


df_mon = df_monitoring.copy()
df_mon_op = df_mon[
    (df_mon["record_type"] == "operator")
    & (df_mon["component_name"] == "cluster-monitoring")
]

rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    op_rows = df_mon_op[df_mon_op["cluster_name"] == cluster_name]
    if op_rows.empty:
        rows.append(
            {
                "cluster_name": cluster_name,
                "cluster_monitoring_installed": False,
                "prometheus_enabled": False,
                "alertmanager_enabled": False,
                "user_workload_monitoring_enabled": False,
            }
        )
        continue
    op = op_rows.iloc[0]
    rows.append(
        {
            "cluster_name": cluster_name,
            "cluster_monitoring_installed": str(op["status"]) == "installed",
            "prometheus_enabled": _detail_has(op["detail_1"], "prometheus=enabled"),
            "alertmanager_enabled": _detail_has(op["detail_2"], "alertmanager=enabled"),
            "user_workload_monitoring_enabled": _detail_has(
                op["detail_3"], "user-workload-monitoring=true"
            ),
        }
    )

df_ocp24_flags = pd.DataFrame(rows)
df_ocp24_flags["compliant"] = df_ocp24_flags[
    [
        "cluster_monitoring_installed",
        "prometheus_enabled",
        "alertmanager_enabled",
        "user_workload_monitoring_enabled",
    ]
].all(axis=1)
df_ocp24_noncompliant = df_ocp24_flags[~df_ocp24_flags["compliant"]].copy()
print(
    f"{len(df_ocp24_noncompliant)} of {len(df_ocp24_flags)} cluster(s) non-compliant for OCP-24"
)
style_table(df_ocp24_noncompliant, caption="OCP-24: Non-compliant clusters")

---
## OCP-25: Configuration Drift Detection

*Drift detection tooling and policies are implemented.*

On-cluster evidence is the state of the **MachineConfigPools** (rolling-update progress and degraded / unavailable counts), **ClusterOperator** availability, **GitOps** (Argo CD / Flux) sync status, **Compliance Operator** scan results, and **node / operator version consistency**. Drift is signalled by any pool not in `Updated`, any operator in `Degraded`, any GitOps app `out-of-sync`, or any version inconsistency across nodes.

### Drift records (per cluster)

In [ ]:
df_drift = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ConfigurationDriftStatus.record_type,
        ConfigurationDriftStatus.component_name,
        ConfigurationDriftStatus.status,
        ConfigurationDriftStatus.namespace,
        ConfigurationDriftStatus.detail_1,
        ConfigurationDriftStatus.detail_2,
        ConfigurationDriftStatus.detail_3,
        ConfigurationDriftStatus.detail_4,
    )
    .join(Cluster, ConfigurationDriftStatus.cluster_id == Cluster.id)
    .order_by(Cluster.cluster_name, ConfigurationDriftStatus.record_type)
    .statement,
    engine,
)
style_table(df_drift, caption="OCP-25: Raw drift / consistency records")

### OCP-25: Compliance flags

Per-cluster flags derived from the raw drift records:

- `gitops_in_use` — at least one `argocd`, `gitops_summary`, `flux_summary`, or `flux_drift` row exists with a non-`not-installed` status.
- `apps_out_of_sync` — sum of `apps-out-of-sync=N` parsed from `argocd` / `gitops_summary` rows.
- `mcp_drifted` — count of `machineconfigpool` rows whose `status` is not `Updated` **or** whose `detail_2` / `detail_3` contains `degraded>0` / `unavailable>0`.
- `degraded_operators` — count of `clusteroperator` rows with `status = Degraded`.

A cluster is considered **non-compliant** if `apps_out_of_sync > 0`, `mcp_drifted > 0`, or `degraded_operators > 0`. `gitops_in_use=False` is a **warning** (no drift detection tooling installed) but listed alongside flagged clusters.

In [ ]:
_OUT_OF_SYNC_RE = re.compile(r"apps-out-of-sync=(\d+)")
_DEGRADED_RE = re.compile(r"degraded=(\d+)")
_UNAVAIL_RE = re.compile(r"unavailable=(\d+)")


def _extract_int(value, regex):
    if value is None:
        return 0
    m = regex.search(str(value))
    return int(m.group(1)) if m else 0


df_d = df_drift.copy()
rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_d[df_d["cluster_name"] == cluster_name]

    gitops_rows = sub[
        sub["record_type"].isin(
            ["argocd", "gitops_summary", "flux_summary", "flux_drift"]
        )
    ]
    gitops_in_use = bool(
        len(gitops_rows[gitops_rows["component_name"] != "not-installed"])
    )

    apps_out_of_sync = 0
    for _, r in sub[sub["record_type"].isin(["argocd", "gitops_summary"])].iterrows():
        for col in ("detail_1", "detail_2", "detail_3", "detail_4"):
            apps_out_of_sync += _extract_int(r[col], _OUT_OF_SYNC_RE)

    mcp_drifted = 0
    for _, r in sub[sub["record_type"] == "machineconfigpool"].iterrows():
        bad_status = str(r["status"]) != "Updated"
        bad_counts = (
            _extract_int(r["detail_2"], _DEGRADED_RE) > 0
            or _extract_int(r["detail_3"], _UNAVAIL_RE) > 0
        )
        if bad_status or bad_counts:
            mcp_drifted += 1

    degraded_operators = int(
        (
            (sub["record_type"] == "clusteroperator") & (sub["status"] == "Degraded")
        ).sum()
    )

    rows.append(
        {
            "cluster_name": cluster_name,
            "gitops_in_use": gitops_in_use,
            "apps_out_of_sync": apps_out_of_sync,
            "mcp_drifted": mcp_drifted,
            "degraded_operators": degraded_operators,
        }
    )

df_ocp25_flags = pd.DataFrame(rows)
df_ocp25_flags["compliant"] = (
    (df_ocp25_flags["apps_out_of_sync"] == 0)
    & (df_ocp25_flags["mcp_drifted"] == 0)
    & (df_ocp25_flags["degraded_operators"] == 0)
)
df_ocp25_noncompliant = df_ocp25_flags[
    ~df_ocp25_flags["compliant"] | ~df_ocp25_flags["gitops_in_use"]
].copy()
print(
    f"{len(df_ocp25_noncompliant)} of {len(df_ocp25_flags)} cluster(s) flagged for OCP-25"
)
style_table(df_ocp25_noncompliant, caption="OCP-25: Flagged clusters")


---
## OCP-26: Centralized Logging, Auditing & Retention

*A unified and secure system for collecting, aggregating, and retaining all security and operational logs is established.*

On-cluster evidence is the **API server audit profile** (Default / WriteRequestBodies / AllRequestBodies) and audit log retention, the **ClusterLogging** operator install posture, the **log forwarder** pipelines (outputs and active state), **audit log forwarding** to an external SIEM, **log retention** periods, and the **LokiStack** size / retention configuration. Record types in scope: `audit_profile`, `cluster_logging`, `log_forwarder`, `log_forwarder_output`, `audit_forwarding`, `log_retention`, `lokistack`.

### Logging & audit records (per cluster)

In [ ]:
OCP26_RECORD_TYPES = (
    "audit_profile",
    "cluster_logging",
    "log_forwarder",
    "log_forwarder_output",
    "audit_forwarding",
    "log_retention",
    "lokistack",
)

df_logging = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        MonitoringAuditLogging.record_type,
        MonitoringAuditLogging.component_name,
        MonitoringAuditLogging.status,
        MonitoringAuditLogging.namespace,
        MonitoringAuditLogging.detail_1,
        MonitoringAuditLogging.detail_2,
        MonitoringAuditLogging.detail_3,
        MonitoringAuditLogging.detail_4,
    )
    .join(Cluster, MonitoringAuditLogging.cluster_id == Cluster.id)
    .filter(MonitoringAuditLogging.record_type.in_(OCP26_RECORD_TYPES))
    .order_by(Cluster.cluster_name, MonitoringAuditLogging.record_type)
    .statement,
    engine,
)

# Also include the cluster-logging operator row from record_type='operator',
# which the export script emits alongside the OCP-26-specific record types.
df_logging_op = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        MonitoringAuditLogging.record_type,
        MonitoringAuditLogging.component_name,
        MonitoringAuditLogging.status,
        MonitoringAuditLogging.namespace,
        MonitoringAuditLogging.detail_1,
        MonitoringAuditLogging.detail_2,
        MonitoringAuditLogging.detail_3,
        MonitoringAuditLogging.detail_4,
    )
    .join(Cluster, MonitoringAuditLogging.cluster_id == Cluster.id)
    .filter(MonitoringAuditLogging.record_type == "operator")
    .filter(MonitoringAuditLogging.component_name == "cluster-logging")
    .statement,
    engine,
)
df_logging_full = pd.concat([df_logging_op, df_logging], ignore_index=True).sort_values(
    ["cluster_name", "record_type"]
)
style_table(df_logging_full, caption="OCP-26: Raw logging & audit records")

### OCP-26: Compliance flags

Per-cluster flags derived from the raw logging / audit records:

- `audit_profile_level` — `status` of the `audit_profile` row (e.g. `Default`, `WriteRequestBodies`, `AllRequestBodies`).
- `audit_log_enabled` — `detail_1` on the `audit_profile` row contains `audit-log-enabled=true`.
- `audit_retention_days` — integer parsed from `retention=Nd` on the `audit_profile` row.
- `cluster_logging_installed` — `operator` row for `cluster-logging` has `status = installed`.
- `forwarder_active` — at least one `log_forwarder` row has `status = active`.

A cluster is considered **non-compliant** when `audit_log_enabled = False`, the audit profile is `Default` (insufficient detail), `audit_retention_days < 30`, `cluster_logging_installed = False`, or `forwarder_active = False`.

In [ ]:
_RETENTION_RE = re.compile(r"retention=(\d+)d")

df_l = df_logging_full.copy()
rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_l[df_l["cluster_name"] == cluster_name]

    audit_rows = sub[sub["record_type"] == "audit_profile"]
    if audit_rows.empty:
        audit_profile_level = None
        audit_log_enabled = False
        audit_retention_days = None
    else:
        a = audit_rows.iloc[0]
        audit_profile_level = a["status"]
        audit_log_enabled = _detail_has(a["detail_1"], "audit-log-enabled=true")
        m = _RETENTION_RE.search(str(a["detail_2"] or ""))
        audit_retention_days = int(m.group(1)) if m else None

    cl_rows = sub[
        (sub["record_type"] == "operator")
        & (sub["component_name"] == "cluster-logging")
    ]
    cluster_logging_installed = (
        not cl_rows.empty and str(cl_rows.iloc[0]["status"]) == "installed"
    )

    fwd_rows = sub[sub["record_type"] == "log_forwarder"]
    forwarder_active = bool((fwd_rows["status"] == "active").any())

    rows.append(
        {
            "cluster_name": cluster_name,
            "audit_profile_level": audit_profile_level,
            "audit_log_enabled": audit_log_enabled,
            "audit_retention_days": audit_retention_days,
            "cluster_logging_installed": cluster_logging_installed,
            "forwarder_active": forwarder_active,
        }
    )

df_ocp26_flags = pd.DataFrame(rows)


def _is_compliant(r):
    if not r["audit_log_enabled"]:
        return False
    if r["audit_profile_level"] == "Default":
        return False
    if r["audit_retention_days"] is None or r["audit_retention_days"] < 30:
        return False
    if not r["cluster_logging_installed"]:
        return False
    if not r["forwarder_active"]:
        return False
    return True


df_ocp26_flags["compliant"] = df_ocp26_flags.apply(_is_compliant, axis=1)
df_ocp26_noncompliant = df_ocp26_flags[~df_ocp26_flags["compliant"]].copy()
print(
    f"{len(df_ocp26_noncompliant)} of {len(df_ocp26_flags)} cluster(s) non-compliant for OCP-26"
)
style_table(df_ocp26_noncompliant, caption="OCP-26: Non-compliant clusters")

In [ ]:
session.close()
print("Session closed.")